# 04 — Model Training and Evaluation

This notebook focuses on developing, training, and evaluating machine learning models for dry-spell warning prediction.

The workflow includes:

- Loading the prepared datasets.
- Establishing a baseline model.
- Evaluating model performance using appropriate classification metrics.
- Comparing models and tuning decision thresholds.
- Selecting a final model based on validation performance.

All experiments will use chronological validation to reflect prediction on future observations.

## 4.1 Environment Setup and Data Loading

This section imports the required libraries and loads the prepared datasets and feature engineering functions.

The data will be reconstructed from the original source files using the project modules to ensure that this notebook can run independently of other notebooks.

### 4.1.1 Imports and Load Data

The original training and testing datasets are loaded using the project's reusable data loading functions.

The datasets are merged and standardized before feature engineering.

In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Set project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project configuration
from config import TRAIN_DIR, TEST_DIR

# Source modules
from src.data_loader import (
    load_and_standardize_datasets,
    merge_datasets
)

from src.feature_engineering import (
    add_lag_and_rolling_features,
    add_calendar_features
)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load and merge datasets
train_datasets = load_and_standardize_datasets(TRAIN_DIR)
test_datasets = load_and_standardize_datasets(TEST_DIR)

train_merged = merge_datasets(train_datasets)
test_merged = merge_datasets(test_datasets)

print("Training data:", train_merged.shape)
print("Testing data:", test_merged.shape)

Training data: (2214, 18)
Testing data: (738, 17)


### 4.1.3 Load Prepared Modeling Data

The prepared modeling datasets are loaded from the processed data directory.

The training and validation sets will be used for model development and evaluation. The test features will remain separate until the final evaluation stage.

In [3]:
processed_path = Path("../data/processed/modeling_data.pkl")

prepared_data = pd.read_pickle(processed_path)

X_train = prepared_data["X_train"]
y_train = prepared_data["y_train"]

X_val = prepared_data["X_val"]
y_val = prepared_data["y_val"]

X_test = prepared_data["X_test"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape)

Training: (1722, 60) (1722,)
Validation: (492, 60) (492,)
Testing: (738, 60)


## 4.2 Baseline Model

A baseline model establishes a simple reference point for evaluating more advanced machine learning models.

A Dummy Classifier will be trained using the most frequent class strategy. Its performance will help demonstrate why accuracy alone is insufficient for this imbalanced classification problem.

In [5]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)

y_val_pred_baseline = baseline_model.predict(X_val)

print("Baseline Model — Validation Results")
print("-----------------------------------")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred_baseline):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_baseline, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred_baseline, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_val, y_val_pred_baseline, zero_division=0):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_baseline))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_baseline, zero_division=0))

Baseline Model — Validation Results
-----------------------------------
Accuracy:  0.9248
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[455   0]
 [ 37   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       455
           1       0.00      0.00      0.00        37

    accuracy                           0.92       492
   macro avg       0.46      0.50      0.48       492
weighted avg       0.86      0.92      0.89       492

